In [59]:

import os
import time
import logging
from dataclasses import dataclass, field
from typing import Optional

import requests
import pandas as pd
from dotenv import load_dotenv
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# If your .env is in a parent directory, point load_dotenv at it explicitly
load_dotenv(os.path.join(os.getcwd(), "..", ".env"))

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

# paste the rest of your class definitions here (everything except __main__)


In [60]:
import sys
import os

# Add the folder containing finbert_sentiment.py to the path
sys.path.insert(0, os.path.join(os.getcwd(), ".."))  # adjust if needed

from src.sentiment import (
    SentimentAnalyzer,
    compute_daily_sentiment,
    compute_weekly_sentiment,
    compute_quarterly_sentiment,
    print_summary,
    GDP_RELEVANT_TOPICS,
    AlphaVantageRateLimitError,
)

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), "..", ".env"))



True

In [66]:
# Base directory
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Data folders
RAW_DATA_DIR       = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, "data", "processed")

# Output folders
FIGURES_DIR        = os.path.join(BASE_DIR, "outputs", "figures")
TABLES_DIR         = os.path.join(BASE_DIR, "outputs", "tables")

In [67]:
analyzer = SentimentAnalyzer(av_api_key=os.getenv("ALPHA_VANTAGE_API_KEY"))
result = analyzer.get_net_sentiment(time_from="20260101T0000", freq="Q")

df = result["df"]
df.to_csv(f"{RAW_DATA_DIR}articles_raw.csv", index=False)
print(f"Saved {len(df)} articles")

19:39:51  INFO      Loading ProsusAI/finbert …
19:39:52  INFO      Model loaded.
19:39:52  INFO      Fetching up to 50 articles  topics=economy_fiscal,economy_macro,economy_monetary,finance,manufacturing,real_estate,retail_wholesale,energy_transportation …
19:39:53  INFO      Fetched 0 articles.
19:39:53  WARNING   Alpha Vantage returned an empty feed.
  Possible causes:
    1. Topic filter too narrow — try fewer topics or topics=None
    2. 'time_from' window has no coverage
  Response keys: ['items', 'sentiment_score_definition', 'relevance_score_definition', 'feed']
19:39:53  WARNING   Topic-filtered fetch returned 0 articles — retrying without topic filter.
19:39:53  INFO      Fetching up to 50 articles  topics=(none) …
19:39:55  INFO      Fetched 50 articles.
19:39:55  INFO      Scoring 50 texts (batch_size=16) …


Saved 50 articles


In [62]:
df = pd.read_csv("articles_raw.csv")
df["published_at"] = pd.to_datetime(df["published_at"])

daily   = compute_daily_sentiment(df)
weekly  = compute_weekly_sentiment(df)
print_summary(df, weekly)

# The number you plug into your GDP model
net_sentiment = result["net_sentiment"]
print("Net sentiment:", net_sentiment)


  FinBERT Sentiment Summary
  Articles analysed : 50
  Positive          : 26.0%
  Neutral           : 64.0%
  Negative          : 10.0%
  Net sentiment     : +0.160  (range −1 to +1)

Top 5 most positive headlines:
  [0.95] Is Korn Ferry (KFY) Still Attractive After Doubling Its Estimated Intrinsic Value Per Shar
  [0.95] United Rentals (URI) Is Up 22.4% After Raising 2026 Revenue Guidance And Posting Record Q1
  [0.95] United Rentals (URI) Is Up 22.4% After Raising 2026 Revenue Guidance And Posting Record Q1
  [0.95] Coterra Energy Delivers Strong Q3 2025 Production Growth and Expanding Free Cash Flow Amid
  [0.95] Sanmina (SANM) Set to Report Q2 Earnings with Strong Growth Expectations

Top 5 most negative headlines:
  [0.97] Iridium Communications (IRDM) Is Down 6.9% After Reaffirming 2026 Outlook Despite Profit S
  [0.94] More than 200,000 '32 Degrees' heated socks from Costco recalled after reports of burn inj
  [0.92] Intuit Stock Rebounds After AI Selloff as TurboTax Owner Nea